# Benchmark de Runtimes PHP — FPM/JIT vs FrankenPHP Worker vs Swoole

Roda a matriz **3 cenários × 3 endpoints** com carga assíncrona (`aiohttp`) e gera tabelas + gráficos comparativos.

**Pré-requisitos:**
1. Ambiente no ar: `docker compose up -d --build`
2. Dependências do notebook: `pip install -r bench/requirements.txt`
3. Abra este notebook (jupyter lab) e rode as células de cima para baixo.

> A célula "rodar ao vivo" leva ~`(DURATION + WARMUP) × (nº cenários × nº endpoints)` segundos.

In [ ]:
import os, sys, glob

# Localiza loadtest.py esteja o notebook aberto em bench/ ou na raiz do projeto
HERE = os.getcwd()
if not os.path.exists(os.path.join(HERE, 'loadtest.py')):
    cand = os.path.join(HERE, 'bench')
    if os.path.exists(os.path.join(cand, 'loadtest.py')):
        HERE = cand
sys.path.insert(0, HERE)

import pandas as pd
import matplotlib.pyplot as plt
from loadtest import run_matrix, save_results, SCENARIOS, ENDPOINTS, RESULTS_DIR

# ----- Parâmetros do teste (ajuste à vontade) -----
DURATION = 20   # segundos por medição
CONN     = 100  # conexões concorrentes
WARMUP   = 5    # segundos descartados (esquenta JIT / workers)
ORDER    = ['fpm', 'frankenphp', 'swoole']  # ordem fixa nas colunas
print('cenários:', list(SCENARIOS), '| endpoints:', list(ENDPOINTS))

## 1. Rodar a matriz ao vivo
Requer o `docker compose` no ar. Salva CSV + JSON em `bench/results/`.

In [ ]:
results = await run_matrix(duration=DURATION, conn=CONN, warmup=WARMUP)
csv_path, json_path = save_results(results)
print('\nsalvo em:', csv_path)
df = pd.DataFrame(results)
df[['scenario', 'endpoint', 'rps', 'lat_avg', 'lat_p50', 'lat_p99', 'non2xx']]

### (Alternativa) Carregar o último resultado salvo
Se não quiser rodar ao vivo, descomente e execute para carregar o CSV mais recente.

In [ ]:
# files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'bench_*.csv')))
# df = pd.read_csv(files[-1])
# print('carregado:', files[-1])
# df

## 2. Tabelas comparativas

In [ ]:
def pivot(metric):
    t = df.pivot_table(index='endpoint', columns='scenario', values=metric)
    return t.reindex(columns=[c for c in ORDER if c in t.columns])

print('Throughput — req/s (MAIOR = melhor)')
display(pivot('rps').round(1))

print('\nLatência p99 — ms (MENOR = melhor)')
display(pivot('lat_p99').round(2))

print('\nLatência média — ms (MENOR = melhor)')
display(pivot('lat_avg').round(2))

print('\nErros (non-2xx)')
display(pivot('non2xx').fillna(0).astype(int))

## 3. Gráficos

In [ ]:
ax = pivot('rps').plot(kind='bar', figsize=(9, 5))
ax.set_title('Throughput por endpoint (req/s) — maior é melhor')
ax.set_ylabel('req/s'); ax.set_xlabel('')
plt.xticks(rotation=0); ax.legend(title='runtime')
for c in ax.containers:
    ax.bar_label(c, fmt='%.0f', fontsize=8, padding=2)
plt.tight_layout(); plt.show()

In [ ]:
ax = pivot('lat_p99').plot(kind='bar', figsize=(9, 5), color=['#c44', '#4a4', '#46c'])
ax.set_title('Latência p99 por endpoint (ms) — menor é melhor')
ax.set_ylabel('ms'); ax.set_xlabel('')
plt.xticks(rotation=0); ax.legend(title='runtime')
plt.tight_layout(); plt.show()

## 4. Speedup relativo ao FPM
Quantas vezes cada runtime é mais rápido (throughput) que o PHP-FPM clássico. `1.00` = igual ao FPM.

In [ ]:
rps = pivot('rps')
if 'fpm' in rps.columns:
    speedup = rps.div(rps['fpm'], axis=0).round(2)
    display(speedup.style.background_gradient(cmap='RdYlGn', axis=None).format('{:.2f}x'))
else:
    print('FPM nao esta nos resultados — sem baseline para speedup.')